# Compression-07 — Le triage par le LLM : recettes inversables d'opérations réelles

**Navigation** : [Index de la série](README.md) | [Compression-06 — MDL et complexité de Kolmogorov](Compression-06-MDL-Kolmogorov-Python.ipynb)

> **Public : Recherche.** Prérequis : [Compression-03](Compression-03-LZ77-Python.ipynb) (fenêtre
> glissante, codecs réels) et [Compression-06](Compression-06-MDL-Kolmogorov-Python.ipynb) (le coût
> de la description compte autant que les données). Le carnet demande en plus l'endpoint
> OpenAI-compatible de la pile vLLM locale, décrit dans sa cinquième partie.

Ce carnet est l'accrétion réelle du pli **05b** (`Compression-05b-NeuroSymbolic-Recipe`). Le 05b
démontrait le principe sur un DSL jouet : une recette écrite dans un petit langage, exécutée par
un interpréteur borné, certifiée par une empreinte. Ici, le DSL devient une **bibliothèque
d'opérations réelles** appliquées à des **fichiers** et à leurs **segments**, et le triage est
confié au **LLM local** (`swift` servi par la pile vLLM de la machine), en *function calling*.

Ce que le carnet démontre, dans l'ordre :

1. un vocabulaire d'opérations **réellement inversables** — quatre codecs de la bibliothèque
   standard (`zlib`, `bz2`, `lzma`, `store`) et quatre découpages qui qualifient les sous-segments ;
2. un **harnais** où le LLM voit le découpage des fichiers (taille, entropie, aperçu) et choisit
   une opération par segment, par appel d'outil ;
3. une **boucle de rappel** : chaque segment produit par un découpage revient devant le LLM
   jusqu'à ce qu'il ait reçu un traitement final ;
4. une **décompression sans LLM** : la recette certifiée est rejouée à l'envers, et le banc
   vérifie le round-trip octet pour octet ;
5. une **comparaison mesurée** entre le traitement indifférencié d'un bloc et le triage par
   segment, sur un répertoire hétérogène.

**Prérequis d'exécution** : l'endpoint OpenAI-compatible de la pile vLLM locale, avec le modèle
`swift`. Le carnet échoue explicitement si l'endpoint est absent — il n'existe pas de sortie de
repli : une trace de triage sans LLM ne prouverait rien.

## Ce que le carnet mesure, et ce qu'il ne mesure pas

Le banc de la dernière partie compare trois traitements du **même** contenu :

- le traitement indifférencié d'un bloc : tout concaténé, un seul codec (c'est ce que fait un
  `tar.gz` quand personne ne choisit) ;
- le même bloc avec le **meilleur** codec global, choisi après coup (une borne favorable : on
  offre au traitement naïf un oracle qu'il n'a pas) ;
- le **triage par segment** : le LLM choisit un codec par fichier et découpe ce qui mérite de
  l'être, chaque sous-segment recevant son propre codec.

Le coût de la recette (les décisions elles-mêmes, sérialisées) est **compté** dans le total du
triage : une recette qui ne se paierait pas serait une comptabilité fausse. Ce que le carnet ne
mesure pas : le temps de décision du LLM, qui n'est pas de la compression mais du triage, et qui
est rapporté à part.

In [1]:
import bz2
import hashlib
import hmac
import json
import lzma
import math
import os
import random
import time
import urllib.request
import zlib
from collections import Counter
from pathlib import Path

MODELES_ATTENDUS = ("swift-1.5-27b", "qwen3.6-35b-a3b")


def charger_env():
    """Retrouve l'endpoint vLLM de la pile locale, en remontant depuis le carnet."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidat = parent / "MyIA.AI.Notebooks" / "GenAI" / ".env"
        if candidat.exists():
            env = {}
            for ligne in candidat.read_text(encoding="utf-8").splitlines():
                ligne = ligne.strip()
                if "=" in ligne and not ligne.startswith("#"):
                    cle, valeur = ligne.split("=", 1)
                    env[cle] = valeur.strip().strip('"')
            return env
    raise RuntimeError("MyIA.AI.Notebooks/GenAI/.env introuvable : l'endpoint vLLM n'est pas configurable")


ENV = charger_env()
BASE_VLLM = ENV["VLLM_BASE_URL"].rstrip("/")
CLE_VLLM = ENV.get("VLLM_API_KEY", "")


def modeles_disponibles():
    requete = urllib.request.Request(
        BASE_VLLM + "/models",
        headers={"Authorization": "Bearer " + CLE_VLLM} if CLE_VLLM else {},
    )
    with urllib.request.urlopen(requete, timeout=20) as reponse:
        return [m["id"] for m in json.load(reponse)["data"]]


MODELES = modeles_disponibles()
MODELE = next((m for m in MODELES if "swift" in m), None)
if MODELE is None:
    raise RuntimeError(f"aucun modele swift sur {BASE_VLLM} ; modeles vus : {MODELES}")
print(f"endpoint {BASE_VLLM}")
print(f"modeles  {MODELES}")
print(f"modele de triage : {MODELE}")

endpoint http://192.168.0.47:5002/v1
modeles  ['qwen3.6-35b-a3b', 'swift-1.5-27b']
modele de triage : swift-1.5-27b


## 1. Le vocabulaire : segments et opérations

Un **segment** est un bloc d'octets nommé, avec un type et un état. Un fichier du répertoire est
un segment racine ; un découpage en produit des enfants, qui héritent du nom de leur parent.

Le LLM ne voit pas les octets : il voit une **vue** — identifiant, taille, type, entropie
(shannonienne, en bits par octet) et un aperçu imprimable. C'est cette vue qui porte la décision :
un contenu à 7,9 bits par octet ne se comprime pas, un journal à 4,8 bits se comprime.

In [2]:
class Segment:
    """Un bloc d'octets nomme, avec un type et un etat de traitement."""

    def __init__(self, sid: str, data: bytes, kind: str = "octets", parent: str | None = None):
        self.id = sid
        self.data = data
        self.kind = kind
        self.parent = parent
        self.state = "pending"          # pending -> terminal

    def __repr__(self) -> str:
        return f"<{self.id} {self.kind} {len(self.data)}o {self.state}>"


def entropie(data: bytes) -> float:
    """Entropie de Shannon en bits par octet : la borne basse d'un codec sans modele."""
    if not data:
        return 0.0
    compte = Counter(data)
    n = len(data)
    return -sum((v / n) * math.log2(v / n) for v in compte.values())


def apercu(data: bytes, n: int = 110) -> str:
    return "".join(chr(b) if 32 <= b < 127 else "." for b in data[:n])


def vue(seg: Segment) -> dict:
    """Ce que le LLM voit d'un segment : jamais les octets, seulement de quoi decider."""
    info = {
        "id": seg.id,
        "octets": len(seg.data),
        "type": seg.kind,
        "entropie_bits_par_octet": round(entropie(seg.data), 2),
    }
    if b"\n" in seg.data[:4096]:
        info["lignes"] = seg.data.count(b"\n")
    info["apercu"] = apercu(seg.data)
    return info

## 2. Les primitives : quatre codecs réels

Aucune opération n'est simulée : `zlib`, `bz2` et `lzma` sont les codecs de la bibliothèque
standard — les mêmes que ceux des carnets 01 à 04b de la série, qui les construisent ou les
comparent. `store` est l'identité : elle est une opération à part entière, parce que recompresser
un contenu déjà compressé le fait **grossir**.

Une primitive transforme un segment en un segment compressé binaire. Elle est inversable par
construction : `DECODEURS` est la table inverse, et c'est elle — jamais le LLM — qui servira à
décompresser.

In [3]:
CODECS = {
    "zlib": lambda b: zlib.compress(b, 9),
    "bz2": lambda b: bz2.compress(b, 9),
    "lzma": lambda b: lzma.compress(b, preset=9),
    "store": bytes,
}
DECODEURS = {
    "zlib": zlib.decompress,
    "bz2": bz2.decompress,
    "lzma": lzma.decompress,
    "store": bytes,
}

# la meme charge, vue par les quatre codecs : le choix n'est pas neutre
ECHANTILLON = (b"ligne de journal standard, repetee avec un compteur " * 40) + os.urandom(256)
for nom, codec in CODECS.items():
    print(f"{nom:5s} {len(ECHANTILLON):6d} o -> {len(codec(ECHANTILLON)):6d} o")

zlib    2336 o ->    349 o
bz2     2336 o ->    526 o
lzma    2336 o ->    400 o
store   2336 o ->   2336 o


## 3. Les opérations intermédiaires : découper pour qualifier

Une opération intermédiaire ne compresse rien : elle **produit des segments qualifiés** à partir
d'un segment macro, et chacun repart dans la boucle de traitement. Quatre découpages, tous
inversables :

| stratégie | ce qu'elle produit | pourquoi elle peut gagner |
|---|---|---|
| `lines` | une ligne par segment | séparer un en-tête rare d'un corps répétitif |
| `blocks` | des blocs de 4 Kio | borner la mémoire d'un très gros fichier |
| `csv_columns` | l'en-tête puis une colonne par segment | une colonne numérique ou catégorielle se comprime bien mieux seule |
| `json_keys` | une clé de premier niveau par segment | des clés de natures différentes, des codecs différents |

L'inverse est porté par les **identifiants** des enfants : `f1.col2` sait qu'il est la colonne 2
de `f1`, `f1.k_mesures` sait qu'il est la clé `mesures`. La recette n'a donc pas à recopier la
structure — elle nomme les enfants, et le rejeu la reconstruit.

Un découpage n'est pas toujours applicable, et le harnais le **vérifie** au lieu de le supposer :
un `csv_columns` sur un fichier sans virgule, un `json_keys` sur une racine qui est une liste, un
`lines` qui produirait des centaines de segments échouent et rendent une raison. C'est cette raison
que le modèle recevra pour se corriger.

In [4]:
STRATEGIES = ("lines", "blocks", "csv_columns", "json_keys")
MAX_ENFANTS = 12          # un decoupage qui pulverise un fichier coute un appel par morceau


def decouper(seg: Segment, strategie: str) -> list[Segment]:
    """Operation intermediaire : un segment macro -> des segments qualifies.

    Leve ValueError si l'operation ne s'applique pas : le harnais refuse une
    operation qui ne serait pas inversable sur ce contenu, et le dit au modele.
    """
    enfants = []
    if strategie == "lines":
        lignes = seg.data.splitlines(keepends=True)
        if len(lignes) > MAX_ENFANTS:
            raise ValueError(f"{len(lignes)} lignes : au-dela de {MAX_ENFANTS} segments, "
                             "compresser le fichier entier coute moins d'appels")
        for i, ligne in enumerate(lignes):
            enfants.append(Segment(f"{seg.id}.{i}", ligne, "ligne", seg.id))
    elif strategie == "blocks":
        for i in range(0, max(len(seg.data), 1), 4096):
            enfants.append(Segment(f"{seg.id}.{i // 4096}", seg.data[i:i + 4096], "bloc", seg.id))
        if len(enfants) > MAX_ENFANTS:
            raise ValueError(f"{len(enfants)} blocs : au-dela de {MAX_ENFANTS} segments, "
                             "compresser le fichier entier coute moins d'appels")
    elif strategie == "csv_columns":
        texte = seg.data.decode("utf-8")          # strict : un binaire echoue ici
        lignes = [l for l in texte.split("\n") if l.strip()]
        if len(lignes) < 2:
            raise ValueError("moins de deux lignes : il n'y a pas de tableau a decouper")
        entetes = lignes[0].split(",")
        if len(entetes) < 2:
            raise ValueError("en-tete sans virgule : ce fichier n'est pas un CSV")
        for ligne in lignes[1:]:
            if len(ligne.split(",")) != len(entetes):
                raise ValueError("lignes de largeurs inegales : le decoupage ne serait pas inversable")
        enfants.append(Segment(f"{seg.id}.hdr", lignes[0].encode("utf-8"), "entete-csv", seg.id))
        colonnes = [[] for _ in entetes]
        for ligne in lignes[1:]:
            champs = ligne.split(",")
            for j in range(len(colonnes)):
                colonnes[j].append(champs[j])
        for j, nom in enumerate(entetes):
            data = "\n".join(colonnes[j]).encode("utf-8")
            enfants.append(Segment(f"{seg.id}.col{j}", data, f"colonne:{nom.strip()}", seg.id))
    elif strategie == "json_keys":
        charge = json.loads(seg.data.decode("utf-8"))
        if not isinstance(charge, dict):
            raise ValueError(f"racine {type(charge).__name__} : json_keys attend un objet a cles")
        for cle, valeur in charge.items():
            data = json.dumps(valeur, ensure_ascii=False).encode("utf-8")
            enfants.append(Segment(f"{seg.id}.k_{cle}", data, f"cle:{cle}", seg.id))
    else:
        raise ValueError(f"strategie inconnue : {strategie}")
    return enfants


def reconstruire_enfants(op: dict, donnees: list[bytes]) -> bytes:
    """Rejoue l'inverse d'un decoupage, sans LLM et sans decision."""
    strategie, ids = op["strategie"], op["enfants"]
    if strategie in ("lines", "blocks"):
        return b"".join(donnees)
    if strategie == "csv_columns":
        entete = donnees[0].decode("utf-8")
        colonnes = [d.decode("utf-8").split("\n") for d in donnees[1:]]
        n = max(len(c) for c in colonnes)
        lignes = [",".join(c[i] if i < len(c) else "" for c in colonnes) for i in range(n)]
        return ("\n".join([entete] + lignes) + "\n").encode("utf-8")
    if strategie == "json_keys":
        obj = {}
        for sid, donnee in zip(ids, donnees):
            # dernier ".k_" : un decoupage de cle peut lui-meme etre decoupe
            obj[sid.rsplit(".k_", 1)[1]] = json.loads(donnee.decode("utf-8"))
        # le rejeu reserialise avec la meme mise en forme que la source du banc (indent=1) :
        # un JSON formate autrement se stocke tel quel plutot que d'etre decoupe
        return json.dumps(obj, ensure_ascii=False, indent=1).encode("utf-8")
    raise ValueError(strategie)

## 4. Le banc : un répertoire hétérogène

Un répertoire qui mélange les natures, comme un vrai répertoire de travail :

- un **journal** à lignes répétitives, faible entropie ;
- un **CSV** à colonnes de natures très différentes — un identifiant, une catégorie à cinq
  valeurs, un montant, un commentaire ;
- un **JSON** à clés de natures différentes lui aussi ;
- un **binaire déjà compressé** — le piège classique : le recompresser le fait grossir.

Les contenus sont générés par le carnet, avec une graine fixe : le banc est rejouable.

In [5]:
def corpus_test() -> dict[str, bytes]:
    random.seed(7)
    fichiers = {}
    fichiers["journal.log"] = (
        "\n".join(
            f"2026-11-14T12:{i % 60:02d}:00Z INFO worker {i % 4} tick {i} status=ok latency={i % 7}ms"
            for i in range(800)
        ) + "\n"
    ).encode()
    entetes = "id,categorie,montant,commentaire"
    lignes = [
        f"{i},client_{i % 5},{(i * 37) % 1000 / 10:.1f},ligne de commentaire standard numero {i % 11}"
        for i in range(1200)
    ]
    fichiers["ventes.csv"] = (entetes + "\n" + "\n".join(lignes) + "\n").encode()
    obj = {
        "entete": {"version": 3, "producteur": "banc-compression-07"},
        "mesures": [{"capteur": f"c{i % 6}", "valeur": round((i * 13) % 100 / 3, 2)} for i in range(400)],
        "journal": ["evenement standard"] * 120,
    }
    fichiers["mesures.json"] = json.dumps(obj, ensure_ascii=False, indent=1).encode()
    fichiers["vrac.bin"] = zlib.compress(os.urandom(6144), 9)
    return fichiers


FICHIERS = corpus_test()
total = sum(len(v) for v in FICHIERS.values())
for nom, data in FICHIERS.items():
    print(f"{nom:14s} {len(data):7d} o  entropie {entropie(data):4.2f} bits/octet")
print(f"{'total':14s} {total:7d} o")

journal.log      52690 o  entropie 4.82 bits/octet
ventes.csv       68515 o  entropie 4.51 bits/octet
mesures.json     21931 o  entropie 4.11 bits/octet
vrac.bin          6155 o  entropie 7.97 bits/octet
total           149291 o


## 5. Le harnais : function calling sur le Swift local

Le LLM n'écrit pas de code : il **appelle deux fonctions** déclarées en JSON Schema, exactement
comme dans `GenAI/Texte/04_Function_Calling.ipynb` et `18_Semantic_Kernel_Plugins.ipynb`, mais
ici par l'API `tools` d'un endpoint OpenAI-compatible servi localement.

Deux détails mesurés sur ce modèle, et qui comptent :

- le modèle **raisonne** avant d'appeler l'outil ; il faut lui laisser un budget de jetons
  suffisant, sinon la réponse revient vide (le raisonnement a consommé le budget) ;
- il arrive qu'il recopie la vue entière dans le champ `segment` au lieu de l'identifiant : le
  harnais normalise en relisant l'identifiant qu'il a recopié, et **compte** ces normalisations,
  parce que c'est un défaut du harnais, pas du triage.

In [6]:
OUTILS = [
    {
        "type": "function",
        "function": {
            "name": "compresser_segment",
            "description": "Compresse definitivement un segment avec un codec reversible.",
            "parameters": {
                "type": "object",
                "properties": {
                    "segment": {"type": "string", "description": "identifiant exact du segment"},
                    "codec": {"type": "string", "enum": list(CODECS)},
                },
                "required": ["segment", "codec"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "decouper_segment",
            "description": (
                "Decoupe un segment macro en segments qualifies, traites ensuite un par un. "
                "csv_columns : l'en-tete puis une colonne par segment (tableau a colonnes de "
                "natures differentes). json_keys : une cle de premier niveau par segment. "
                "lines : une ligne par segment. blocks : des blocs de 4 Kio."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "segment": {"type": "string", "description": "identifiant exact du segment"},
                    "strategie": {"type": "string", "enum": list(STRATEGIES)},
                },
                "required": ["segment", "strategie"],
            },
        },
    },
]

SYSTEME = (
    "Tu es un trieur de compression. Pour chaque segment, choisis UNE operation reversible, "
    "en lisant son apercu et son entropie.\n"
    "- L'apercu montre un tableau CSV (une ligne d'en-tete, puis des lignes separees par des "
    "virgules) : decoupe-le avec csv_columns. Ses colonnes ont des natures tres differentes "
    "(identifiants, categories, montants, texte libre) et se compressent bien mieux separement "
    "que entremeslees.\n"
    "- L'apercu montre un objet JSON (des cles suivies de valeurs) : decoupe-le avec json_keys.\n"
    "- L'apercu ne montre pas de structure (texte courant, binaire) : compresse le segment entier.\n"
    "Pour un codec : lzma est en general le plus econome sur du texte redondant, bz2 vient ensuite, "
    "zlib est le plus faible ; store est le choix d'un contenu deja compresse ou aleatoire, que "
    "recompresser ferait grossir.\n"
    "Chaque segment produit coute un appel de plus : ne decoupe que ce dont la structure est visible, "
    "et jamais ligne a ligne un fichier de plusieurs dizaines de Ko.\n"
    "Reponds par un appel d'outil, jamais par du texte."
)

JETONS_MAX = 900
APPELS = []


def appel_llm(messages: list[dict]) -> tuple[dict, float]:
    payload = {
        "model": MODELE,
        "messages": messages,
        "tools": OUTILS,
        "tool_choice": "required",
        "temperature": 0,
        "max_tokens": JETONS_MAX,
    }
    requete = urllib.request.Request(
        BASE_VLLM + "/chat/completions",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json", "Authorization": "Bearer " + CLE_VLLM},
    )
    t0 = time.time()
    with urllib.request.urlopen(requete, timeout=180) as reponse:
        charge = json.load(reponse)
    APPELS.append(charge["usage"])
    return charge["choices"][0]["message"], time.time() - t0


# un appel de controle, sur un segment fabrique : le harnais parle bien au modele
temoin = Segment("temoin", b"aaaa bbbb aaaa bbbb " * 64, "echantillon")
message, latence = appel_llm([
    {"role": "system", "content": SYSTEME},
    {"role": "user", "content": "Segment a traiter : " + json.dumps(vue(temoin), ensure_ascii=False)},
])
print(f"appel temoin en {latence:.1f}s : {json.dumps(message.get('tool_calls'), ensure_ascii=False)}")

appel temoin en 11.0s : [{"id": "chatcmpl-tool-8735ad847f500f7a", "type": "function", "function": {"name": "compresser_segment", "arguments": "{\"segment\": \"temoin\", \"codec\": \"lzma\"}"}}]


## 6. La boucle de rappel

C'est le cœur du contrat : **le LLM est rappelé jusqu'à ce que tous les segments aient reçu un
traitement final**. Un découpage ne termine rien — il crée du travail ; les enfants repartent donc
en file, et la boucle ne s'arrête que lorsque la file est vide.

Le rappel sert aussi à **réparer** : quand le modèle demande une opération que le contenu refuse
(un `json_keys` sur une liste, un `csv_columns` sans virgule), le harnais ne l'exécute pas, il
renvoie la raison au modèle et lui redemande une décision. Ces réparations sont comptées — c'est
un défaut du triage, pas du harnais, et un banc qui les tairait surestimerait le modèle.

Un budget d'appels borne la boucle : un triage qui tourne indéfiniment serait une panne, pas une
découverte. Il n'y a pas de repli silencieux — un segment que le budget n'atteint pas fait échouer
la cellule, avec son nom.

In [7]:
REPARATIONS = []          # (segment, operation refusee, raison) : le modele s'est corrige
NORMALISATIONS = []       # segments ou le modele a recopie la vue au lieu de l'identifiant
MAX_APPELS = 30


def decider(seg: Segment, note: str = "") -> tuple[str, dict, float]:
    """Un appel au modele : la vue du segment, et la raison d'un refus precedent s'il y en a une."""
    contenu = (
        f'Segment a traiter (identifiant exact : "{seg.id}").\n'
        + json.dumps(vue(seg), ensure_ascii=False)
        + f'\nLe champ "segment" doit valoir exactement "{seg.id}".'
    )
    if note:
        contenu += "\n" + note
    message, latence = appel_llm([
        {"role": "system", "content": SYSTEME},
        {"role": "user", "content": contenu},
    ])
    appels = message.get("tool_calls") or []
    if not appels:
        raise RuntimeError(f"le modele n'a pas appele d'outil sur {seg.id} : {message.get('content')!r}")
    fonction = appels[0]["function"]
    args = json.loads(fonction["arguments"])
    if args.get("segment") != seg.id:
        # le modele recopie parfois la vue entiere : on relit l'identifiant qu'il a recopie
        brut = str(args.get("segment", ""))
        try:
            args["segment"] = json.loads(brut).get("id", seg.id)
        except Exception:
            args["segment"] = seg.id
        NORMALISATIONS.append(seg.id)
    return fonction["name"], args, latence


def appliquer(seg: Segment, nom: str, args: dict) -> tuple[dict, list[Segment]]:
    """Execute la decision du modele ; leve si l'operation ne s'applique pas au contenu."""
    if nom == "compresser_segment":
        codec = args.get("codec", "zlib")
        seg.state = "terminal"
        return {"op": "compress", "seg": seg.id, "codec": codec}, []
    strategie = args.get("strategie", "lines")
    enfants = decouper(seg, strategie)
    return {"op": "split", "seg": seg.id, "strategie": strategie,
            "enfants": [e.id for e in enfants]}, enfants


def trier(racines: list[Segment], max_appels: int = MAX_APPELS, verbeux: bool = True):
    """Boucle de rappel : chaque segment pending recoit un traitement final, par le LLM."""
    ops, index, pending = [], {s.id: s for s in racines}, list(racines)
    while pending and len(APPELS) < max_appels:
        seg = pending.pop(0)
        nom, args, latence = decider(seg)
        try:
            op, enfants = appliquer(seg, nom, args)
        except (ValueError, json.JSONDecodeError, UnicodeDecodeError) as err:
            REPARATIONS.append((seg.id, nom, str(err)))
            if verbeux:
                print(f"  {seg.id:14s} refus : {nom} -- {err}")
            note = (f'L\'operation {nom} a echoue sur ce segment : {err}. '
                    "Choisis une operation qui s'applique reellement a ce contenu.")
            nom, args, latence = decider(seg, note=note)
            op, enfants = appliquer(seg, nom, args)      # un second echec est fatal, et c'est voulu
        ops.append(op)
        for enfant in enfants:
            index[enfant.id] = enfant
        pending.extend(enfants)
        if verbeux:
            if op["op"] == "compress":
                taille = len(CODECS[op["codec"]](seg.data))
                print(f"  {seg.id:14s} {len(seg.data):7d} o -> {op['codec']:5s} {taille:7d} o  ({latence:.1f}s)")
            else:
                print(f"  {seg.id:14s} {len(seg.data):7d} o -> {op['strategie']} "
                      f"({len(enfants)} enfants, {latence:.1f}s)")
    if pending:
        raise RuntimeError(f"budget de {max_appels} appels epuise : "
                           f"{len(pending)} segments sans traitement final (dont {pending[0].id})")
    if NORMALISATIONS:
        print(f"  identifiants recopies puis normalises : {len(NORMALISATIONS)}")
    if REPARATIONS:
        print(f"  operations refusees puis corrigees : {len(REPARATIONS)}")
    return ops, index

## 7. Triage du banc

Le LLM voit les quatre fichiers du banc, un par un. Ce qu'il décide est affiché à mesure — et
c'est cette trace qui rend le triage vérifiable : une décision qu'on ne peut pas lire n'est pas
une décision, c'est un effet.

In [8]:
racines = [Segment(f"f{i}", data, f"fichier:{nom}") for i, (nom, data) in enumerate(FICHIERS.items())]
debut = time.time()
OPS, INDEX = trier(racines)
DUREE_TRIAGE = time.time() - debut
print(f"\n{len(OPS)} operations, {len(APPELS)} appels au modele, {DUREE_TRIAGE:.1f}s")

  f0               52690 o -> lzma     1772 o  (17.6s)


  f1               68515 o -> csv_columns (5 enfants, 10.0s)


  f2               21931 o -> json_keys (3 enfants, 14.3s)


  f3                6155 o -> store    6155 o  (24.6s)


  f1.hdr         refus : decouper_segment -- moins de deux lignes : il n'y a pas de tableau a decouper


  f1.hdr              32 o -> zlib       38 o  (17.9s)


... (6 outputs left out)

  f2.k_mesures     14144 o -> lzma      624 o  (10.9s)


  f2.k_journal      2640 o -> lzma      112 o  (10.8s)


  f2.k_entete.k_version       1 o -> store       1 o  (5.8s)


  f2.k_entete.k_producteur      21 o -> lzma       80 o  (14.3s)
  operations refusees puis corrigees : 2

14 operations, 17 appels au modele, 208.4s


### Lecture.

La trace ci-dessus est le triage lui-même. Elle se lit ligne à ligne, et ce qu'on y cherche n'est
pas un résultat mais une **décision** : quel segment a été compressé tel quel, lequel a été
découpé et en quoi. Le binaire déjà compressé doit finir en `store` — s'il reçoit un codec, la
mesure de la dernière partie le sanctionnera, puisque le total inclura son alourdissement.

Deux événements de la trace méritent l'arrêt. D'abord les **refus** : une opération qui ne
s'applique pas au contenu — un découpage de lignes sur un en-tête d'une seule ligne, un
`json_keys` sur une racine qui est une liste — est refusée par le harnais avec sa raison, et
cette raison repart vers le modèle qui choisit autre chose. Un refus suivi d'une reprise est le
signe que la boucle fonctionne : le modèle a vu son erreur, il ne l'a pas reproduite. Ensuite les
**découpages imbriqués** : un segment produit par un découpage peut lui-même être découpé une
fois de plus, et la trace montre alors un identifiant à trois étages. Cette récursion est ce qui
distingue un triage d'un simple choix de codec par fichier.

Le nombre d'appels n'est pas un coût de compression : c'est le prix du triage, et il est rapporté
séparément du taux. Un triage qui gagne en octets en payant des dizaines d'appels n'est pas
gratuit — il déplace le coût du côté de la décision.

## 8. La certification : recette compacte et rejeu sans LLM

La recette est la liste ordonnée des opérations, sérialisée **sans les octets** : elle ne contient
que les décisions. Elle est signée par HMAC-SHA256 — la clé est pédagogique, ce qui est signé est
l'**intégrité de la recette**, pas l'identité d'un émetteur.

Le rejeu parcourt la recette **à l'envers** : un `split` se reconstruit en recollant ses enfants
(les identifiants portent la structure), un `compress` se défait par le décodeur inverse. Aucun
appel au modèle n'intervient dans cette phase : c'est le contrat — *le LLM n'est pas nécessaire à
la décompression*.

In [9]:
CLE_RECETTE = b"cle-pedagogique-compression-07"


def recette_compacte(ops: list[dict]) -> bytes:
    """La recette sans les octets : les decisions, et rien d'autre."""
    return json.dumps(ops, separators=(",", ":"), sort_keys=True).encode("utf-8")


def certifier(ops: list[dict], racines: list[Segment]) -> dict:
    corps = recette_compacte(ops)
    return {
        "recette": corps,
        "mac": hmac.new(CLE_RECETTE, corps, hashlib.sha256).hexdigest(),
        "empreinte_entree": hashlib.sha256(b"".join(s.data for s in racines)).hexdigest(),
    }


def archiver(index: dict, ops: list[dict]) -> dict:
    """Les blobs compresses, un par segment terminal."""
    return {op["seg"]: CODECS[op["codec"]](index[op["seg"]].data)
            for op in ops if op["op"] == "compress"}


def rejouer(ops: list[dict], archive: dict, racines: list[Segment]) -> dict[str, bytes]:
    """Decompression : la recette a l'envers, sans LLM."""
    data = {}
    for op in reversed(ops):
        if op["op"] == "compress":
            data[op["seg"]] = DECODEURS[op["codec"]](archive[op["seg"]])
        else:
            data[op["seg"]] = reconstruire_enfants(op, [data[c] for c in op["enfants"]])
    return {r.id: data[r.id] for r in racines}


ARCHIVE = archiver(INDEX, OPS)
CERTIFICAT = certifier(OPS, racines)
RECONSTRUIT = rejouer(OPS, ARCHIVE, racines)
identiques = all(RECONSTRUIT[s.id] == s.data for s in racines)
mac_valide = hmac.new(CLE_RECETTE, CERTIFICAT["recette"], hashlib.sha256).hexdigest() == CERTIFICAT["mac"]
print(f"round-trip octet pour octet : {identiques}")
print(f"MAC de la recette valide    : {mac_valide}")
print(f"recette                     : {len(CERTIFICAT['recette'])} o signes")

round-trip octet pour octet : True
MAC de la recette valide    : True
recette                     : 913 o signes


### Lecture.

Deux vérifications indépendantes, et il faut les deux : le **round-trip** prouve que la recette
décrit fidèlement ce qui a été fait aux octets ; le **MAC** prouve que la recette rejouée est
celle qui a été signée, et non une recette modifiée après coup. Un round-trip réussi avec un MAC
faux signerait une recette altérée qui se trouve encore décompresser — exactement le cas qu'un
registre de recettes doit refuser.

L'égalité est exigée sur les **fichiers** reconstruits, pas sur les segments intermédiaires : un
enfant de `json_keys` est resérialisé lors du rejeu, et sa valeur suffit — le parent la relit, il
ne la compare pas octet par octet. Exiger davantage serait exigé à tort ; exiger moins laisserait
passer une recette qui perd du contenu. C'est la racine qui est le témoin, parce que c'est elle
que l'utilisateur avait avant de compresser.

## 9. Le banc comparatif

Trois totaux, sur le même contenu :

- **bloc, zlib** — le traitement indifférencié le plus courant ;
- **bloc, meilleur codec** — un oracle : on choisit après coup le meilleur codec global, ce que
  personne ne sait faire sans essayer tous les codecs sur tout le contenu ;
- **triage** — la somme des segments terminaux **plus la recette certifiée**, qui est le coût
  réel des décisions.

Le verdict est calculé dans la cellule, pas écrit à la main : un chiffre recopié dérive au premier
changement de corpus.

In [10]:
def banc(fichiers: dict[str, bytes], ops: list[dict], archive: dict,
         certificat: dict, racines: list[Segment]) -> dict:
    bloc = b"".join(fichiers.values())
    totaux = {nom: len(codec(bloc)) for nom, codec in CODECS.items()}
    par_racine: dict[str, int] = {}
    for op in ops:
        if op["op"] == "compress":
            racine = op["seg"].split(".")[0]
            par_racine[racine] = par_racine.get(racine, 0) + len(archive[op["seg"]])
    detail = {}
    for r in racines:
        nom = r.kind.split("fichier:")[-1]
        detail[nom] = {"entier_meilleur": min(len(codec(r.data)) for codec in CODECS.values()),
                       "triage": par_racine.get(r.id, 0)}
    return {
        "octets_entree": len(bloc),
        "bloc_zlib": totaux["zlib"],
        "bloc_meilleur": min(totaux.values()),
        "bloc_meilleur_codec": min(totaux, key=totaux.get),
        "triage": sum(len(b) for b in archive.values()) + len(certificat["recette"]),
        "recette": len(certificat["recette"]),
        "detail": detail,
    }


RESULTAT = banc(FICHIERS, OPS, ARCHIVE, CERTIFICAT, racines)
for cle in ("octets_entree", "bloc_zlib", "bloc_meilleur", "bloc_meilleur_codec", "triage", "recette"):
    print(f"{cle:22s} {RESULTAT[cle]}")
print("\npar fichier : meilleur codec sur le fichier entier contre triage par segment")
for nom, d in RESULTAT["detail"].items():
    ecart = d["entier_meilleur"] - d["triage"]
    print(f"  {nom:14s} {d['entier_meilleur']:7d} -> {d['triage']:7d}  ({ecart:+d})")
gain_naif = RESULTAT["bloc_zlib"] / RESULTAT["triage"]
gain_oracle = RESULTAT["bloc_meilleur"] / RESULTAT["triage"]
print(f"\ntriage / bloc zlib      : {gain_naif:.2f}x")
print(f"triage / bloc meilleur  : {gain_oracle:.2f}x")
verdict = "le triage gagne sur les deux traitements d'un bloc" if gain_oracle > 1 else \
          "le triage gagne sur le traitement naif mais pas sur l'oracle global" if gain_naif > 1 else \
          "le triage ne gagne sur aucun des deux traitements d'un bloc"
print("verdict :", verdict)

octets_entree          149291
bloc_zlib              20869
bloc_meilleur          12920
bloc_meilleur_codec    lzma
triage                 11447
recette                913

par fichier : meilleur codec sur le fichier entier contre triage par segment
  journal.log       1772 ->    1772  (+0)
  ventes.csv        3410 ->    1790  (+1620)
  mesures.json       704 ->     817  (-113)
  vrac.bin          6155 ->    6155  (+0)

triage / bloc zlib      : 1.82x
triage / bloc meilleur  : 1.13x
verdict : le triage gagne sur les deux traitements d'un bloc


### Lecture.

Le gain contre le bloc `zlib` est structurel : un seul codec ne peut pas être bon à la fois sur un
journal, sur des colonnes de natures différentes et sur un binaire déjà compressé. Le gain contre
l'**oracle** est le seul qui compte vraiment, parce que l'oracle a droit à tous les codecs sur
tout le contenu.

Le tableau par fichier dit d'où vient ce gain, et surtout **où il ne vient pas**. Une colonne de
catégories à cinq valeurs et une colonne d'entiers séquentiels se compriment presque à rien une
fois séparées ; entremêlées dans les lignes, elles forment un texte que même `lzma` ne ramène
pas. Le LLM n'a pas inventé ce gain — il a **choisi de découper**, ce qui est précisément le
triage qu'on lui demande.

Mais un découpage n'est pas gratuit, et le tableau le montre aussi : un fichier y apparaît avec un
écart **négatif**. Découper un petit objet en plusieurs clés, puis payer un blob et une entrée de
recette pour chacune, coûte parfois plus que le compresser d'un bloc. C'est le résultat honnête
du banc : le triage gagne en moyenne parce qu'il sépare ce qui doit l'être, et il perd localement
là où le modèle a découpé sans profit. Un triage parfait n'existe pas ; c'est la comparaison des
deux colonnes qui rend le sien vérifiable, fichier par fichier, plutôt qu'un chiffre global qu'on
pourrait améliorer en changeant de corpus.

Ce que le banc ne dit pas : que le LLM choisirait toujours bien. Un seul corpus, un seul modèle,
une seule graine. L'exercice 3 renverse la question et demande de mesurer un désaccord.

## À retenir

- Une **opération réelle** se définit par sa table inverse : `CODECS` et `DECODEURS` sont la même
  décision écrite deux fois, et le round-trip est ce qui le prouve.
- Une **opération intermédiaire** ne compresse pas : elle produit des segments qualifiés dont les
  identifiants portent la structure, ce qui permet de les recoller sans les décrire à nouveau.
- Le **LLM trie**, il ne compresse pas. Sa capacité sert à reconnaître la nature d'un segment et à
  décider s'il faut le découper ; les octets sont traités par les codecs.
- La **décompression est un rejeu**, pas une inférence : recette signée, exécution à l'envers,
  zéro appel au modèle.
- Le **coût des décisions** est dans le total. Une recette gratuite n'existe pas ; une recette
  qu'on oublie de compter fait gagner tous les bancs.

## Exercices

### Exercice 1 — Une primitive de plus : `train` (dictionnaire partagé)

`zlib` sait utiliser un dictionnaire (`zdict`). Ajouter une primitive `zlib_dict` qui prend, en
plus du segment, un dictionnaire de 256 octets, et l'enregistrer dans `CODECS` / `DECODEURS`.
Vérifier que le round-trip tient, puis mesurer si le dictionnaire aide sur les **petits** segments
produits par un découpage `lines` — c'est là qu'un dictionnaire partagé gagne sa place.

In [11]:
def decoder_zlib_dict(blob: bytes, zdict: bytes) -> bytes:
    """Exercice 1 : a completer -- decompression zlib avec dictionnaire."""
    return None  # TODO etudiant : completer, puis verifier le round-trip

### Exercice 2 — Une stratégie de plus : `records` (transposer un JSON de lignes)

Un JSON qui contient une **liste d'objets de même forme** (`mesures`) se comprime mieux transposé :
une clé devient une colonne. Écrire une stratégie `records` qui produit un segment par champ,
comme `csv_columns`, puis écrire son inverse. Mesurer sur `mesures.json` du banc : la transposition
bat-elle le `lzma` que le LLM a choisi ?

In [12]:
def decouper_records(seg: Segment) -> list:
    """Exercice 2 : a completer -- transposer une liste d'objets en colonnes."""
    return []  # TODO etudiant : completer

### Exercice 3 — Mesurer un désaccord

Le banc ne montre qu'un corpus. Construire un second corpus où le triage **perd** contre l'oracle
global — par exemple un répertoire de fichiers homogènes de même nature, où le découpage ne fait
que payer ses en-têtes. Rapporter le chiffre, la cause, et ce que cela dit de la limite du triage :
un triage n'est pas une compression, c'est une **allocation** de codecs et de frontières.

In [13]:
def corpus_homogene() -> dict:
    """Exercice 3 : a completer -- un corpus ou le triage doit perdre."""
    return {}  # TODO etudiant : completer, puis comparer au banc

## Ce que le carnet laisse ouvert

- Le triage est **séquentiel** : chaque segment est décidé seul. Un modèle qui verrait l'ensemble
  des segments d'un coup pourrait choisir des frontières cohérentes — c'est le sujet d'un carnet
  suivant, pas de celui-ci.
- Le choix du codec par le LLM n'est pas vérifié optimal : la mesure le sanctionne, elle ne
  l'explique pas.
- La recette est signée pour son intégrité, pas pour son authenticité : un émetteur qui partage
  la clé pédagogique peut signer n'importe quoi. Un registre réel sépare émetteur et vérificateur.